In [1]:
import pandas as pd
import numpy as np
#matplotlib widget
import plotly # built with plotly version '4.4.1'
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from plotly.subplots import make_subplots
import sys
sys.path.append('C:/Users/wg984/Wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data

In [2]:
# result:
# - interactive full data
# - interactive day night partitions
# - static full data
# - static day night partitions

# best case, they share underlying function.

In [2]:
filepath = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data_daynight/icusleep_001_d02.h5'
# filepath = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data/icusleep_001.h5'

In [3]:
data, hdr = load_sleep_data(filepath, idx_to_datetime=1)

In [4]:
data.head()

,ART1D,ART1S,CUFF,HR,NBPD,NBPS,SPO2%,accx,accy,accz,...,ventilation0,ventilation_10s,ventilation_3s,ventilation_60s,ventmax_10s,ventmax_30s,ventmax_60s,ventmedian_10s,ventmedian_30s,ventmedian_60s
2018-06-06 08:00:00.000,48.90625,103.2500,NaN,80.3125,NaN,NaN,95.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-06 08:00:00.100,49.03125,103.3750,NaN,80.3750,NaN,NaN,95.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-06 08:00:00.200,49.12500,103.5625,NaN,80.4375,NaN,NaN,95.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-06 08:00:00.300,49.25000,103.7500,NaN,80.5000,NaN,NaN,95.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-06 08:00:00.400,49.34375,103.8750,NaN,80.5625,NaN,NaN,95.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
data.index[3]

Timestamp('2018-06-06 08:00:00.300000', freq='100L')

In [11]:
data.index[6]

Timestamp('2018-06-06 08:00:00.600000', freq='100L')

In [13]:
data.loc[data.index[3]:data.index[6]].index

DatetimeIndex(['2018-06-06 08:00:00.300000', '2018-06-06 08:00:00.400000',
               '2018-06-06 08:00:00.500000', '2018-06-06 08:00:00.600000'],
              dtype='datetime64[ns]', freq='100L')

In [5]:
data.columns

Index(['ART1D', 'ART1S', 'CUFF', 'HR', 'NBPD', 'NBPS', 'SPO2%', 'accx', 'accy',
       'accz', 'band', 'deriv1', 'deriv2', 'hypo_10_60', 'movavg_10s',
       'movavg_1_2s', 'movavg_1s', 'movavg_2s', 'movavg_3s', 'movavg_5s',
       'movavg_60s', 'movmedian_10min', 'movmedian_5min', 'movstd_10min',
       'movstd_10s', 'movstd_30s', 'movstd_5min', 'movstd_60s', 'peaks',
       'rr_10s', 'rr_60s', 'ventilation0', 'ventilation_10s', 'ventilation_3s',
       'ventilation_60s', 'ventmax_10s', 'ventmax_30s', 'ventmax_60s',
       'ventmedian_10s', 'ventmedian_30s', 'ventmedian_60s'],
      dtype='object')

In [14]:
def icu_sleep_plot(data, trace_linewidth=1, labelfontsize=10, legend_position = [0.2, 1.15], legend_font_size=10):
    '''
    Plot designed for the 10Hz data, plots AirGo, Blood Pressure, Heart Rate, and SpO2.
    Input: sleep research-formatted data
    Output: fig (plotly figure instance)
    '''
    
    airgo_available = 'band' in data.columns
    BP_available = any(['ART1D' in data.columns, 'ART1S' in data.columns, 'NBPD' in data.columns, 'NBPS' in data.columns])
    HR_available = 'HR' in data.columns
    SPO2_available = 'SPO2%' in data.columns
    num_traces = sum([airgo_available, BP_available, HR_available, SPO2_available])

    fig = make_subplots(rows=num_traces, cols=1, shared_xaxes=True, shared_yaxes=False, specs=[[{"secondary_y": False}]] * num_traces )

    traces = []
    i_trace=1

    if airgo_available:
        trace_AirGo = go.Scatter(x=data.index, y=data.band, name = 'AirGo', hoverinfo = 'x+y', line = dict(color = 'dodgerblue', width = trace_linewidth), opacity = 0.8 )
        fig.add_trace(trace_AirGo, i_trace, 1)
        fig.update_yaxes(title_text="Circumference <br> Thorax (a.u.)", row=i_trace, col=1, range=[data['band'].quantile(0.01),data['band'].quantile(0.99)], title_font=dict(size=labelfontsize))
        i_trace+=1

    if 'ART1D' in data.columns:
        trace_ART1D = go.Scatter(x=data.index, y=data.ART1D, name = 'Art. BP diastolic', hoverinfo = 'x+y', line = dict(color = 'magenta', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_ART1D, i_trace, 1)
    if 'ART1S' in data.columns:
        trace_ART1S = go.Scatter(x=data.index, y=data.ART1S, name = 'Art. BP systolic', hoverinfo = 'x+y', line = dict(color = 'orange', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_ART1S, i_trace, 1)
    if 'NBPD' in data.columns:
        trace_NBPD = go.Scatter(x=data.index, y=data.NBPD, name = 'NonInv. BP diastolic', hoverinfo = 'x+y', line = dict(color = 'magenta', width = trace_linewidth, dash = 'dashdot'), opacity = 0.8 )
        fig.add_trace(trace_NBPD, i_trace, 1)
    if 'NBPS' in data.columns:
        trace_NBPS = go.Scatter(x=data.index, y=data.NBPS, name = 'NonInv. BP systolic', hoverinfo = 'x+y', line = dict(color = 'orange', width = trace_linewidth, dash = 'dashdot'), opacity = 0.8 )
        fig.add_trace(trace_NBPS, i_trace, 1)
    if BP_available: 
        fig.update_yaxes(title_text="BP (mmHg)", row=i_trace, col=1, title_font=dict(size=labelfontsize))
        i_trace+=1

    if HR_available:
        trace_HR = go.Scatter(x=data.index, y=data.HR, name = 'Heart Rate', hoverinfo = 'x+y', line = dict(color = 'crimson', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_HR, i_trace, 1)
        fig.update_yaxes(title_text="HR(bpm)", row=i_trace, col=1, range=[data['HR'].quantile(0.01),data['HR'].quantile(0.99)], title_font=dict(size=labelfontsize))
        i_trace+=1

    if SPO2_available:
        trace_SPO2 = go.Scatter(x=data.index, y=data['SPO2%'], name = 'SpO2%', hoverinfo = 'x+y', line = dict(color = 'navy', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_SPO2, i_trace, 1)
        fig.update_yaxes(title_text="SpO2 (%)", row=i_trace, col=1, range=[data['SPO2%'].quantile(0.01),data['SPO2%'].quantile(1)], title_font=dict(size=labelfontsize))
        i_trace+=1

    fig.update_layout(plot_bgcolor='rgb(237,237,237)')     # 'white'
    fig.update_layout(legend=dict(x=legend_position[0], y=legend_position[1], orientation = 'h'), font=dict(size=legend_font_size, color="black"))
    return fig



In [11]:
if 'day_night_id' in hdr: 
    title = f'StudyID '+str(hdr['study_id']).zfill(3)+', '+hdr['day_night_id'].split('_')[1].replace('n', 'Night ').replace('d','Day')
    save_path = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_plots/FILEFORMAT_day_night_partitions/'+hdr['day_night_id']
else: 
    title = f'StudyID '+str(hdr['study_id']).zfill(3)+', Full Data'
    save_path = 'M:/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_plots/FILEFORMAT_full/'+str(hdr['study_id']).zfill(3)


In [12]:
# PLOT (html version)
fig = icu_sleep_plot(data, trace_linewidth=1, labelfontsize=12, legend_position = [0.2, 1.05], legend_font_size=12)
fig.update_layout(title=title)
plot(fig, filename=save_path.replace('FILEFORMAT','HTML')+'.html', auto_open=False)
del fig

In [13]:
# PLOT (pdf/png version)
fig = icu_sleep_plot(data, trace_linewidth=0.5, labelfontsize=8, legend_position = [0.18, 1.15], legend_font_size=8)
fig.update_layout(title=title)
fig.write_image(save_path.replace('FILEFORMAT','PDF')+'.pdf', scale=3)
fig.write_image(save_path.replace('FILEFORMAT','PNG')+'.png', scale=5)
del fig

In [ ]:
fig.write_image(save_path.replace('FILEFORMAT','PDF')+'.pdf', scale=3)


In [ ]:
fig.write_image(save_path.replace('FILEFORMAT','PNG')+'.png', scale=5)


In [ ]:
fig